## Load Data

In [22]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/Detailed_Statistics_Departures.csv")
df.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,DL,1/1/26,312.0,N814NW,HNL,10:10,11:20,539.0,554.0,70.0,12:13,53.0,70.0,0.0,15.0,0.0,0.0
1,DL,1/1/26,377.0,N581DA,ANC,15:44,15:42,365.0,384.0,-2.0,16:11,29.0,0.0,0.0,17.0,0.0,0.0
2,DL,1/1/26,762.0,N320NB,PSC,19:25,19:23,209.0,228.0,-2.0,20:02,39.0,0.0,0.0,17.0,0.0,0.0
3,DL,1/1/26,858.0,N350NA,SFO,18:55,18:51,265.0,255.0,-4.0,19:19,28.0,0.0,0.0,0.0,0.0,0.0
4,DL,1/1/26,868.0,N935DZ,SEA,8:39,8:41,246.0,253.0,2.0,9:20,39.0,0.0,0.0,0.0,0.0,0.0


## Cleaning metadata information
Removing rows where date is missing

In [23]:
df = df[df["Date (MM/DD/YYYY)"].notna()]
df = df[~df["Date (MM/DD/YYYY)"].astype(str).str.contains("SOURCE", case=False, na=False)]

## Renaming Columns

In [25]:
df = df.rename(columns={
    "Carrier Code": "carrier_code",
    "Date (MM/DD/YYYY)": "flight_date",
    "Flight Number": "flight_number",
    "Tail Number": "tail_number",
    "Destination Airport": "destination_airport",
    "Scheduled departure time": "scheduled_departure_time",
    "Actual departure time": "actual_departure_time",
    "Scheduled elapsed time (Minutes)": "scheduled_elapsed_minutes",
    "Actual elapsed time (Minutes)": "actual_elapsed_minutes",
    "Departure delay (Minutes)": "departure_delay_minutes",
    "Wheels-off time": "wheels_off_time",
    "Taxi-Out time (Minutes)": "taxi_out_minutes",
    "Delay Carrier (Minutes)": "carrier_delay_minutes",
    "Delay Weather (Minutes)": "weather_delay_minutes",
    "Delay National Aviation System (Minutes)": "nas_delay_minutes",
    "Delay Security (Minutes)": "security_delay_minutes",
    "Delay Late Aircraft Arrival (Minutes)": "late_aircraft_delay_minutes"
})

## Date Conversion

In [26]:
df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce")

/var/folders/9g/5jsrvl5j3033d7jqgz2gmgxw0000gn/T/ipykernel_64557/652859262.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce")


In [27]:
print(df["flight_date"].head())
print("Missing dates:", df["flight_date"].isna().sum())
print("Minimum date:", df["flight_date"].min())
print("Maximum date:", df["flight_date"].max())

0   2026-01-01
1   2026-01-01
2   2026-01-01
3   2026-01-01
4   2026-01-01
Name: flight_date, dtype: datetime64[ns]
Missing dates: 0
Minimum date: 2026-01-01 00:00:00
Maximum date: 2026-05-31 00:00:00


## Converting Numeric Columns

In [28]:
numeric_cols = [
    "flight_number",
    "scheduled_elapsed_minutes",
    "actual_elapsed_minutes",
    "departure_delay_minutes",
    "taxi_out_minutes",
    "carrier_delay_minutes",
    "weather_delay_minutes",
    "nas_delay_minutes",
    "security_delay_minutes",
    "late_aircraft_delay_minutes"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## Creating Time-based Features

In [29]:
df["flight_year"] = df["flight_date"].dt.year
df["flight_month"] = df["flight_date"].dt.month
df["flight_month_name"] = df["flight_date"].dt.month_name()
df["flight_day"] = df["flight_date"].dt.day
df["flight_day_name"] = df["flight_date"].dt.day_name()
df["is_weekend"] = df["flight_day_name"].isin(["Saturday", "Sunday"])

## Extracting Departure Hour

In [30]:
df["scheduled_departure_time_clean"] = df["scheduled_departure_time"].astype(str).str.strip()

df["scheduled_departure_hour"] = (
    df["scheduled_departure_time_clean"]
    .str.split(":")
    .str[0]
    .astype(int)
)

df["scheduled_departure_minute"] = (
    df["scheduled_departure_time_clean"]
    .str.split(":")
    .str[1]
    .astype(int)
)

## Creating flight delay flags
Industry standard: flight is delayed if delay is more than 15 minutes.

In [31]:
df["is_departure_delayed"] = df["departure_delay_minutes"] > 15
df["is_early_departure"] = df["departure_delay_minutes"] < 0
df["is_on_time"] = df["departure_delay_minutes"].between(0, 15)

## Creating flight delay categories

In [32]:
def delay_category(delay):
    if pd.isna(delay):
        return "Unknown"
    elif delay < 0:
        return "Early"
    elif delay <= 15:
        return "On Time"
    elif delay <= 60:
        return "Minor Delay"
    elif delay <= 120:
        return "Moderate Delay"
    else:
        return "Severe Delay"

df["delay_category"] = df["departure_delay_minutes"].apply(delay_category)

In [33]:
def classify_departure_period(hour):
    if pd.isna(hour):
        return "Unknown"
    elif 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"


df["scheduled_departure_period"] = (
    df["scheduled_departure_hour"]
    .apply(classify_departure_period)
)

In [34]:
df.head()

,carrier_code,flight_date,flight_number,tail_number,destination_airport,scheduled_departure_time,actual_departure_time,scheduled_elapsed_minutes,actual_elapsed_minutes,departure_delay_minutes,wheels_off_time,taxi_out_minutes,carrier_delay_minutes,weather_delay_minutes,nas_delay_minutes,security_delay_minutes,late_aircraft_delay_minutes,flight_year,flight_month,flight_month_name,flight_day,flight_day_name,is_weekend,scheduled_departure_time_clean,scheduled_departure_hour,scheduled_departure_minute,is_departure_delayed,is_early_departure,is_on_time,delay_category,scheduled_departure_period
0,DL,2026-01-01,312.0,N814NW,HNL,10:10,11:20,539.0,554.0,70.0,12:13,53.0,70.0,0.0,15.0,0.0,0.0,2026,1,January,1,Thursday,False,10:10,10,10,True,False,False,Moderate Delay,Morning
1,DL,2026-01-01,377.0,N581DA,ANC,15:44,15:42,365.0,384.0,-2.0,16:11,29.0,0.0,0.0,17.0,0.0,0.0,2026,1,January,1,Thursday,False,15:44,15,44,False,True,False,Early,Afternoon
2,DL,2026-01-01,762.0,N320NB,PSC,19:25,19:23,209.0,228.0,-2.0,20:02,39.0,0.0,0.0,17.0,0.0,0.0,2026,1,January,1,Thursday,False,19:25,19,25,False,True,False,Early,Evening
3,DL,2026-01-01,858.0,N350NA,SFO,18:55,18:51,265.0,255.0,-4.0,19:19,28.0,0.0,0.0,0.0,0.0,0.0,2026,1,January,1,Thursday,False,18:55,18,55,False,True,False,Early,Evening
4,DL,2026-01-01,868.0,N935DZ,SEA,8:39,8:41,246.0,253.0,2.0,9:20,39.0,0.0,0.0,0.0,0.0,0.0,2026,1,January,1,Thursday,False,8:39,8,39,False,False,True,On Time,Morning


## Total delay caused in minutes

In [35]:
delay_cause_cols = [
    "carrier_delay_minutes",
    "weather_delay_minutes",
    "nas_delay_minutes",
    "security_delay_minutes",
    "late_aircraft_delay_minutes"
]

df[delay_cause_cols] = df[delay_cause_cols].fillna(0)

df["total_cause_delay_minutes"] = df[delay_cause_cols].sum(axis=1)

## Saving the cleaned-up file

In [36]:
df.to_csv("../data/processed/flights_clean.csv", index=False)

In [37]:
df.columns.tolist()

['carrier_code',
 'flight_date',
 'flight_number',
 'tail_number',
 'destination_airport',
 'scheduled_departure_time',
 'actual_departure_time',
 'scheduled_elapsed_minutes',
 'actual_elapsed_minutes',
 'departure_delay_minutes',
 'wheels_off_time',
 'taxi_out_minutes',
 'carrier_delay_minutes',
 'weather_delay_minutes',
 'nas_delay_minutes',
 'security_delay_minutes',
 'late_aircraft_delay_minutes',
 'flight_year',
 'flight_month',
 'flight_month_name',
 'flight_day',
 'flight_day_name',
 'is_weekend',
 'scheduled_departure_time_clean',
 'scheduled_departure_hour',
 'scheduled_departure_minute',
 'is_departure_delayed',
 'is_early_departure',
 'is_on_time',
 'delay_category',
 'scheduled_departure_period',
 'total_cause_delay_minutes']